In [ ]:
import cvxpy as cp
import numpy as np
from itertools import product
from itertools import combinations
import time

In [ ]:
def cal_pow(R, beta, h_val):
    sort_arr = []
    for j in range(len(beta)):
        sort_arr.append((beta[j]/h_val[j], j))
    sorted_list = [y[1] for y in sorted(sort_arr, key=lambda x: x[0])]
    pow_vals = np.zeros(len(beta))
    for i in range(len(beta)):
        idx = sorted_list[i]
        sum_i_M = np.sum([R[k] for k in range(idx, len(beta))])
        sum_ip1_M = np.sum([R[k] for k in range(min(idx+1, len(beta)), len(beta))])
        pow = (2**(sum_i_M) - 2**(sum_ip1_M))/h_val[idx]
        pow_vals[idx] = pow
    return pow_vals

v_k = np.array([0.1, 0.1, 0.1])
alpha = 0.2
max_iter = 1000
for itr in range(max_iter):
    print('itr', itr)
    t_start = time.time()
    B = v_k 
    n_user = 3
    h_bad =  0.1
    h_good = 1
    pr_h_bad = 0.5
    pr_h_good = (1- pr_h_bad)
    r_max = 2
    h_vec_u1 = np.array([h_bad,  h_good]).tolist()
    permutations_list = []
    permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
    combined_h_vec = []
    for permutation in permutations_list:
        arr = []
        for zz in permutation:
            arr.append(zz)
        combined_h_vec.append(arr)
    p_h_u1 = np.array([pr_h_bad, pr_h_good])
    permutations_list = []
    permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
    pr_h = []
    for permutation in permutations_list:
        arr = []
        for k in permutation:
            arr.append(k)
        pr_h.append(np.prod(arr))
    hp = pr_h
    h = combined_h_vec
    P_bar, l = 2, 0.5
    pkt_prob = n_user*[l]
    weights = n_user*[1/n_user]
    rho = np.arange(0, r_max+1, 1)
    P_bar_U = n_user*[P_bar]
    W = list(product(rho, repeat=n_user))
    print(W)
    mu = cp.Variable((len(h), len(W) ), nonneg=True)
    p = []
    for x in range(n_user):
        temp  = 0
        for k in range(len(h)):
            for j in range(len(W)):
                if W[j][x] != 0:
                    temp+= mu[k][j] * hp[k]
        p.append(temp)
    P_u = []
    for x in range(n_user):
        temp = 0
        for k in range(len(h)):
            for j in range( len(W)):
                temp+= cal_pow(W[j], B, h[k])[x] * mu[k][j] *  hp[k]
        P_u.append(temp)
    obj_fun = 0
    for n in range(n_user):
        obj_fun += (weights[n] *  pkt_prob[n] * (cp.inv_pos(p[n]) - 1) 
                    + B[n] * (P_u[n]-P_bar_U[n]))
    objective = cp.Minimize(obj_fun)
    constraints = []
    a_val = []
    for q in range(len(h)):
        a_val.append( cp.sum([ mu[q][j] for j in range(len(W))]))
    for a in a_val:
        constraints.append(a == 1)
    for i in range(len(h)):           
        for j in range(len(W)):
            constraints.append(0<= mu[i][j])
            constraints.append(mu[i][j] <= 1)
    prob = cp.Problem(objective, constraints)
    prob.solve()
    print('age', np.round(prob.value, 4))
    mu_opt = mu.value
    P_u_g = []
    for x in range(n_user):
        temp = 0
        for k in range(len(h)):
            for j in range( len(W)):
                temp+= cal_pow(W[j], B, h[k])[x] * mu_opt[k][j] *  hp[k]
        P_u_g.append(temp)
    print('pow', np.round(P_u_g, 3))
    subgradient = []
    for n in range(n_user):
        subgradient.append(P_u_g[n] - P_bar_U[n])
    print(subgradient)
    eta_k = alpha / np.sqrt(itr + 1)
    for n in range(n_user):
        v_k[n] = max(0, v_k[n] + eta_k * subgradient[n])
    print('dual var', np.round( v_k, 3))
    print('-------------------------------------')

    print(time.time() - t_start)

# Time sharing

In [1]:
import cvxpy as cp
import numpy as np
from itertools import product
from itertools import combinations
import time

In [2]:
n_user = 3
h_bad =  0.1
h_good = 1
pr_h_bad = 0.5
pr_h_good = (1- pr_h_bad)
r_max = 2
rho = np.arange(0, r_max+1, 1)
h_vec_u1 = np.array([h_bad,  h_good]).tolist()
permutations_list = []
permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
combined_h_vec = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    combined_h_vec.append(arr)
p_h_u1 = np.array([pr_h_bad, pr_h_good])
permutations_list = []
permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
pr_h = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    pr_h.append(np.prod(arr))
hp = pr_h
h = combined_h_vec
W = list(product(rho, repeat=n_user))
rho_DO = []
# poss_dec_ord = [(1, 2), (2, 1)]
poss_dec_ord = [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
for w in W:
    for DO in poss_dec_ord:
        rho_DO.append([w, DO])
def cal_pow(R, DO, h_val):
    sorted_list = DO
    pow_vals = np.zeros(len(DO))
    for i in range(len(DO)):
        idx = DO[i]-1
        sum_i_M = np.sum([R[DO[k]-1]  for k in range(i, len(DO))])
        sum_ip1_M = np.sum([R[DO[k]-1] for k in range(min(i+1, len(DO)), len(DO))])
        pow = (2**(sum_i_M) - 2**(sum_ip1_M))/h_val[idx]
        pow_vals[idx] = pow
    return pow_vals

P_bar, D_bar, l = 3, 1, 1
D_bar_U = n_user*[D_bar]
pkt_prob = n_user*[l]
weights = n_user*[1]
P_bar_U = n_user*[P_bar]
mu = cp.Variable((len(h), len(rho_DO) ), nonneg=True)
p = []
for x in range(n_user):
    temp  = 0
    for k in range(len(h)):
        for j in range(len(rho_DO)):
            if rho_DO[j][0][x] != 0:
                temp+= mu[k][j] * hp[k]
    p.append(temp)
P_u = []
for x in range(n_user):
    temp = 0
    for k in range(len(h)):
        for j in range( len(rho_DO)):
            temp+= cal_pow(rho_DO[j][0], rho_DO[j][1] , h[k])[x] * mu[k][j] *  hp[k]
    P_u.append(temp)
D_u = []
for x in range(n_user):
    temp = 0
    for k in range(len(h)):
        for j in range( len(rho_DO)):
            bit = rho_DO[j][0][x]
            # temp+= (np.exp(-bit) * mu[k][j] *  hp[k] * int(bit > 0))
            temp+= ((1 - bit/r_max)**(2) * mu[k][j] *  hp[k] * int(bit > 0))
    D_u.append(temp)
obj_fun = 0
for n in range(n_user):
    obj_fun = obj_fun + ( weights[n] *  pkt_prob[n] * (cp.inv_pos(p[n])  -  1 )  )
objective = cp.Minimize(obj_fun)
constraints = []
for n in range(n_user):
    # constraints.append(P_u[n] <= P_bar_U[n] )
    val1 = pkt_prob[n] * (1- p[n])+ p[n]
    constraints.append(P_u[n]*pkt_prob[n] <= P_bar_U[n]*val1 )
    constraints.append(D_u[n]*pkt_prob[n] <= D_bar_U[n] * val1)
a_val = []
for q in range(len(h)):
    a_val.append( cp.sum([ mu[q][j] for j in range(len(rho_DO))]))
for a in a_val:
    constraints.append(a == 1)
for i in range(len(h)):           
    for j in range(len(rho_DO)):
        constraints.append(0<= mu[i][j])
        constraints.append(mu[i][j] <= 1)
prob = cp.Problem(objective, constraints)
prob.solve()

print('age', np.round(prob.value, 4))
# print('opt mu', np.round(mu.value, 3))

print(np.round((P_u[0].value, P_u[1].value, P_u[2].value), 3))

/home/sysad/.local/lib/python3.10/site-packages/cvxpy/problems/problem.py:158: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn("Objective contains too many subexpressions. "
/home/sysad/.local/lib/python3.10/site-packages/cvxpy/problems/problem.py:164: UserWarning: Constraint #0 contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn(f"Constraint #{i} contains too many subexpressions. "
/home/sysad/.local/lib/python3.10/site-packages/cvxpy/problems/problem.py:164: UserWarning: Constraint #1 contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn(f"Constraint #{i} contains too many subexpressions. "
/home/sysad/.local/lib/python3.10/site-packages/cvxpy/problems/problem.py:164: UserWarning: Constraint #2 contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compil

age 1.2936
[3. 3. 3.]


In [ ]:
def cal_pow(R, beta, h_val):
    sort_arr = []
    for j in range(len(beta)):
        sort_arr.append((beta[j]/h_val[j], j))
    sorted_list = [y[1] for y in sorted(sort_arr, key=lambda x: x[0])]
    pow_vals = np.zeros(len(beta))
    for i in range(len(beta)):
        idx = sorted_list[i]
        sum_i_M = np.sum([R[k] for k in range(idx, len(beta))])
        sum_ip1_M = np.sum([R[k] for k in range(min(idx+1, len(beta)), len(beta))])
        pow = (2**(sum_i_M) - 2**(sum_ip1_M))/h_val[idx]
        pow_vals[idx] = pow
    return pow_vals

B = [1, 1, 1]
n_user = 3
h_bad =  0.1
h_good = 1
pr_h_bad = 0.5
pr_h_good = (1- pr_h_bad)
r_max = 2
h_vec_u1 = np.array([h_bad,  h_good]).tolist()
permutations_list = []
permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
combined_h_vec = []
for permutation in permutations_list:
    arr = []
    for zz in permutation:
        arr.append(zz)
    combined_h_vec.append(arr)
p_h_u1 = np.array([pr_h_bad, pr_h_good])
permutations_list = []
permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
pr_h = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    pr_h.append(np.prod(arr))
hp = pr_h
h = combined_h_vec
P_bar, D_bar, l = 2, 0.06, 0.5
pkt_prob = n_user*[l]
weights = n_user*[1/n_user]
rho = np.arange(0, r_max+1, 1)
P_bar_U= n_user*[P_bar]
D_bar_U = n_user*[D_bar]
W = list(product(rho, repeat=n_user))
print(W)
mu = cp.Variable((len(h), len(W) ), nonneg=True)
p = []
for x in range(n_user):
    temp  = 0
    for k in range(len(h)):
        for j in range(len(W)):
            if W[j][x] != 0:
                temp+= mu[k][j] * hp[k]
    p.append(temp)
P_u = []
for x in range(n_user):
    temp = 0
    for k in range(len(h)):
        for j in range( len(W)):
            temp+= cal_pow(W[j], B, h[k])[x] * mu[k][j] *  hp[k]
    P_u.append(temp)
D_u = []
for x in range(n_user):
    temp = 0
    for k in range(len(h)):
        for j in range( len(W)):
            bit = W[j][x]
            temp+= ((1 - bit/r_max)**(2) * mu[k][j] *  hp[k] * int(bit > 0))
    D_u.append(temp)
obj_fun = 0
for n in range(n_user):
    obj_fun += (weights[n] *  pkt_prob[n] * (cp.inv_pos(p[n]) - 1) )
objective = cp.Minimize(obj_fun)
constraints = []
for n in range(n_user):
    # constraints.append(P_u[n] <= P_bar_U[n] )
    val1 = pkt_prob[n] * (1- p[n])+ p[n]
    constraints.append(P_u[n]*pkt_prob[n] <= P_bar_U[n]*val1 )
    constraints.append(D_u[n]*pkt_prob[n] <= D_bar_U[n] * val1)
a_val = []
for q in range(len(h)):
    a_val.append( cp.sum([ mu[q][j] for j in range(len(W))]))
for a in a_val:
    constraints.append(a == 1)
for i in range(len(h)):           
    for j in range(len(W)):
        constraints.append(0<= mu[i][j])
        constraints.append(mu[i][j] <= 1)
prob = cp.Problem(objective, constraints)
prob.solve()
print('age', np.round(prob.value, 4))
print(np.round((P_u[0].value, P_u[1].value, P_u[2].value), 3))